## No-crop feature extraction, several encoders in one pass (TCGA and PAIP)

Each non-white 512x512 patch is read and decoded once and handed **whole** (no crop) to every selected encoder, through that encoder's own transform. One `float32` vector `(D,)` is saved per patch and encoder.

Per encoder, model loading, transform and precision are those of its five-crop run. This was verified on 2026-10-09 by recomputing stored five-crop features of both cohorts (cosine ≥ 0.9999996):

| encoder | input | precision | output |
|---|---|---|---|
| `uni2` | timm transform, Resize 224 (bilinear) | fp32, `no_grad` | 1536 |
| `h_optimus_1` | `Resize((224,224))`, H-optimus mean/std | fp16 autocast | 1536 |
| `conch_v1` | CONCH preprocess, Resize 448 (bicubic) | fp32, `no_grad` | 512 (`encode_image`, no projection) |
| `conch_v1_5` | TITAN `return_conch()`, Resize 448 (bilinear) | fp32, `no_grad` | 768 |
| `virchow2` | `Resize((224,224))`, ImageNet mean/std | fp16 autocast | 2560 (class token + mean patch token) |

Output: `<cohort>/Features/<Model>_nocrop/<slide>/<patch>.pt`. Only patches in `<cohort>/final_merged_without_white.csv` are processed. Runs are resumable per patch and encoder, and files are written atomically.

**Run order:** restart the kernel, run every cell from the top, then the run cell(s) at the end. Run one cohort at a time, because both use the GPU. Expect about 3.5 h for PAIP with all five encoders and about 10 h for TCGA with the three remaining ones; the GPU is the limit.

In [ ]:
import os
# The gated models load from the local Hugging Face cache. This must be set before
# torchvision / timm / transformers import huggingface_hub, so run this cell first after a kernel restart.
os.environ["HF_HUB_OFFLINE"] = "1"

import sys
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, List

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

REPO = Path(r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier")
sys.path.insert(0, str(REPO / "feature_extraction"))      # repo-local `conch` package
sys.path.insert(0, str(REPO / "slide_classification"))    # config.paths
from config import paths as P

CONCH_V1_CKPT = REPO / "feature_extraction" / "checkpoints" / "CONCH" / "pytorch_model.bin"
COHORTS = {
    "tcga": P.BASE_ROOT / "dataset",
    "paip": P.BASE_ROOT / "paip_data",
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, {k: str(v) for k, v in COHORTS.items()})

In [ ]:
@dataclass
class Encoder:
    name: str
    dim: int
    fp16: bool           # fp16 autocast + inference_mode; otherwise fp32 + no_grad
    net: torch.nn.Module
    transform: Callable  # PIL image -> (3, H, W)
    call: Callable       # (net, batch) -> raw output, run inside the precision context
    post: Callable       # raw output -> (B, dim), run after it

    def forward(self, batch: torch.Tensor) -> torch.Tensor:
        batch = batch.to(DEVICE, non_blocking=True)
        if self.fp16:
            with torch.inference_mode():
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
                    out = self.call(self.net, batch)
                out = self.post(out)
        else:
            with torch.no_grad():
                out = self.post(self.call(self.net, batch))
        return out.float().cpu()


def _identity(x):
    return x


def load_uni2() -> Encoder:
    import timm
    from timm.data import resolve_data_config
    from timm.data.transforms_factory import create_transform

    kw = dict(img_size=224, patch_size=14, depth=24, num_heads=24, init_values=1e-5, embed_dim=1536,
              mlp_ratio=2.66667 * 2, num_classes=0, no_embed_class=True, mlp_layer=timm.layers.SwiGLUPacked,
              act_layer=torch.nn.SiLU, reg_tokens=8, dynamic_img_size=True)
    net = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **kw).to(DEVICE).eval()
    tf = create_transform(**resolve_data_config(net.pretrained_cfg, model=net))
    return Encoder("uni2", 1536, False, net, tf, lambda n, x: n(x), _identity)


def load_h_optimus_1() -> Encoder:
    import timm
    from torchvision import transforms

    net = timm.create_model("hf-hub:bioptimus/H-optimus-1", pretrained=True,
                            init_values=1e-5, dynamic_img_size=False).to(DEVICE).eval()
    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517)),
    ])
    return Encoder("h_optimus_1", 1536, True, net, tf, lambda n, x: n(x), _identity)


def load_conch_v1() -> Encoder:
    from conch.open_clip_custom import create_model_from_pretrained

    net, preprocess = create_model_from_pretrained("conch_ViT-B-16", str(CONCH_V1_CKPT))
    return Encoder("conch_v1", 512, False, net.to(DEVICE).eval(), preprocess,
                   lambda n, x: n.encode_image(x, proj_contrast=False, normalize=False), _identity)


def load_conch_v1_5() -> Encoder:
    from transformers import AutoModel

    titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
    net, tf = titan.return_conch()
    return Encoder("conch_v1_5", 768, False, net.to(DEVICE).eval(), tf, lambda n, x: n(x), _identity)


def load_virchow2() -> Encoder:
    import timm
    from timm.data import resolve_data_config
    from timm.layers import SwiGLUPacked
    from torchvision import transforms

    net = timm.create_model("hf-hub:paige-ai/Virchow2", pretrained=True,
                            mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU).to(DEVICE).eval()
    cfg = resolve_data_config(net.pretrained_cfg, model=net)
    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=cfg["mean"], std=cfg["std"]),
    ])

    def post(out):  # (B, 261, 1280): token 0 = class, 1-4 = registers, 5+ = patches
        return torch.cat([out[:, 0], out[:, 5:].mean(dim=1)], dim=-1)

    return Encoder("virchow2", 2560, True, net, tf, lambda n, x: n(x), post)


LOADERS: Dict[str, Callable[[], Encoder]] = {
    "uni2": load_uni2,
    "h_optimus_1": load_h_optimus_1,
    "conch_v1": load_conch_v1,
    "conch_v1_5": load_conch_v1_5,
    "virchow2": load_virchow2,
}
OUTPUT_FOLDERS = {"uni2": "UNI2H_nocrop", "h_optimus_1": "H_Optimus_1_nocrop", "conch_v1": "Conch_v1_nocrop",
                  "conch_v1_5": "Conch_v1_5_nocrop", "virchow2": "Virchow2_nocrop"}

In [ ]:
@dataclass
class Job:
    patch_path: str
    save_paths: Dict[str, str]   # encoder -> .pt path, only for encoders still missing this patch


def build_jobs(cohort_root: Path, out_root: Path, models: List[str], slide_names=None):
    """One job per non-white patch that at least one selected encoder still lacks."""
    names = pd.read_csv(cohort_root / "final_merged_without_white.csv", usecols=["patch_name"])
    valid = set(names["patch_name"].astype(str))
    patch_root = cohort_root / "patch_data"
    slides = sorted(d.name for d in os.scandir(patch_root) if d.is_dir())
    if slide_names is not None:
        missing = set(slide_names) - set(slides)
        if missing:
            raise ValueError(f"unknown slide(s): {sorted(missing)}")
        slides = [s for s in slides if s in set(slide_names)]

    jobs, stats = [], {m: {"expected": 0, "done": 0} for m in models}
    for slide in tqdm(slides, desc="Listing slides", unit="slide"):
        pngs = sorted(f.name for f in os.scandir(patch_root / slide) if f.name in valid)
        if not pngs:
            tqdm.write(f"  no non-white patches for {slide}")
            continue
        existing = {}
        for m in models:
            d = out_root / OUTPUT_FOLDERS[m] / slide
            d.mkdir(parents=True, exist_ok=True)
            existing[m] = {f.name[:-3] for f in os.scandir(d) if f.name.endswith(".pt")}
        for png in pngs:
            stem = png[:-4]
            need = {}
            for m in models:
                stats[m]["expected"] += 1
                if stem in existing[m]:
                    stats[m]["done"] += 1
                else:
                    need[m] = str(out_root / OUTPUT_FOLDERS[m] / slide / f"{stem}.pt")
            if need:
                jobs.append(Job(str(patch_root / slide / png), need))
    return jobs, stats

In [ ]:
def save_atomic(tensor: torch.Tensor, path: str) -> None:
    # Write to .tmp, then rename, so an interrupted run never leaves a half-written .pt
    tmp = path + ".tmp"
    torch.save(tensor, tmp)
    os.replace(tmp, path)


def forward_safe(enc: Encoder, batch: torch.Tensor) -> torch.Tensor:
    """Forward a batch; on CUDA out-of-memory, retry in two halves (recursively)."""
    try:
        return enc.forward(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            raise
        tqdm.write(f"  [OOM] {enc.name}: batch of {len(batch)} split in two")
        half = len(batch) // 2
        return torch.cat([forward_safe(enc, batch[:half]), forward_safe(enc, batch[half:])])


def extract(jobs: List[Job], encoders: Dict[str, Encoder], batch_size: int, num_workers: int) -> Dict[str, int]:
    failed = {m: 0 for m in encoders}

    def load(job: Job) -> Dict[str, torch.Tensor]:
        img = Image.open(job.patch_path).convert("RGB")   # decoded once for all encoders
        return {m: encoders[m].transform(img) for m in job.save_paths}

    prefetch = batch_size * 4
    with ThreadPoolExecutor(num_workers) as loader, ThreadPoolExecutor(2) as saver:
        pending, saving = deque(), deque()
        job_iter = iter(jobs)

        def submit_next():
            job = next(job_iter, None)
            if job is not None:
                pending.append((job, loader.submit(load, job)))

        def collect_saves(max_queued):
            while saving and (saving[0][2].done() or len(saving) > max_queued):
                m, path, fut = saving.popleft()
                try:
                    fut.result()
                except Exception as e:
                    failed[m] += 1
                    tqdm.write(f"  [save failed] {m} {path}: {e}")

        for _ in range(prefetch):
            submit_next()

        pbar = tqdm(total=len(jobs), desc="Extracting (" + ", ".join(encoders) + ")", unit="patch")
        while pending:
            batch = []   # (job, {encoder: tensor})
            while pending and len(batch) < batch_size:
                job, fut = pending.popleft()
                submit_next()
                try:
                    batch.append((job, fut.result()))
                except Exception as e:
                    for m in job.save_paths:
                        failed[m] += 1
                    pbar.update(1)
                    tqdm.write(f"  [load failed] {job.patch_path}: {e}")
            if not batch:
                continue

            for m, enc in encoders.items():
                rows = [(job, x[m]) for job, x in batch if m in job.save_paths]
                if not rows:
                    continue
                try:
                    feats = forward_safe(enc, torch.stack([x for _, x in rows]))   # (b, D)
                except Exception as e:
                    failed[m] += len(rows)
                    tqdm.write(f"  [forward failed] {m}, {len(rows)} patches: {e}")
                    continue
                for (job, _), f in zip(rows, feats):
                    # clone(): a row view would otherwise save the whole (b, D) batch storage
                    saving.append((m, job.save_paths[m], saver.submit(save_atomic, f.clone(), job.save_paths[m])))
            collect_saves(max_queued=prefetch * len(encoders))
            pbar.update(len(batch))

        collect_saves(max_queued=0)
        pbar.close()
    return failed

In [ ]:
_LOADED: Dict[str, Encoder] = {}   # encoders stay loaded between calls in this kernel


def run_nocrop(cohort: str, models, slides=None, out_root=None,
               batch_size: int = 64, num_workers: int = 16, check_only: bool = False):
    """Extract no-crop features for `models` ('all' or a list) on `cohort` ('tcga' or 'paip').

    slides     : list of slide folders (default: all)
    out_root   : where the <Model>_nocrop folders go (default: <cohort>/Features)
    check_only : report completeness only, load no model
    """
    models = list(LOADERS) if models == "all" else list(models)
    unknown = [m for m in models if m not in LOADERS]
    if unknown:
        raise ValueError(f"unknown model(s) {unknown}; choose from {list(LOADERS)}")

    cohort_root = COHORTS[cohort]
    out_root = Path(out_root) if out_root else cohort_root / "Features"
    jobs, stats = build_jobs(cohort_root, out_root, models, slides)
    for m, s in stats.items():
        print(f"  {m:12s} -> {out_root / OUTPUT_FOLDERS[m]}: {s['done']:,} of {s['expected']:,} patches done")
    print(f"{len(jobs):,} patches still needed by at least one encoder")
    if check_only or not jobs:
        return

    encoders = {}
    for m in models:
        if any(m in j.save_paths for j in jobs):
            if m not in _LOADED:
                print(f"loading {m} ...", flush=True)
                _LOADED[m] = LOADERS[m]()
            encoders[m] = _LOADED[m]
    if DEVICE.type == "cuda":
        print(f"models on GPU: {torch.cuda.memory_allocated() / 2**30:.1f} GB", flush=True)

    failed = extract(jobs, encoders, batch_size, num_workers)
    for m in encoders:
        n = sum(m in j.save_paths for j in jobs)
        print(f"  {m:12s}: {n - failed[m]:,} saved, {failed[m]:,} failed")
    if any(failed.values()):
        print("Run the same cell again to retry the failures.")

## Check completeness (no GPU)

Counts the saved files against the non-white CSV. Safe to run at any time, including while nothing is loaded.

In [ ]:
run_nocrop("paip", "all", check_only=True)
run_nocrop("tcga", ["conch_v1", "conch_v1_5", "virchow2"], check_only=True)

## Quick test on one slide (optional)

Writes into a separate `_nocrop_test` folder so the real output folders stay untouched. Delete that folder afterwards.

In [ ]:
run_nocrop("paip", "all", slides=["training_data_46_nonMSIH"],
           out_root=COHORTS["paip"] / "Features" / "_nocrop_test")

## PAIP: all five encoders (~3.5 h)

In [ ]:
run_nocrop("paip", "all")

## TCGA: CONCH v1, CONCH v1.5, Virchow2 (~10 h)

UNI2 and H-Optimus-1 no-crop already exist for TCGA (`UNI2H_nocrop`, `H_Optimus_1_nocrop`).

In [ ]:
run_nocrop("tcga", ["conch_v1", "conch_v1_5", "virchow2"])